In [1]:
import json
import pathlib
import re

from tqdm.auto import tqdm

In [2]:
RAW_DATA_DIR = pathlib.Path("raw_data")

RETRIEVED_DIR = RAW_DATA_DIR / "retrieved"
EXTRACTED_DIR = RAW_DATA_DIR / "extracted"
assert EXTRACTED_DIR.exists() and RETRIEVED_DIR.exists()

TREATED_DATA_DIR = pathlib.Path("treated_data")
BLOCKS_DIR = TREATED_DATA_DIR / "data_blocks"
BLOCKS_DIR.mkdir(parents=True, exist_ok=True)

EMPTY_BLOCKS_FILE = TREATED_DATA_DIR / "empty_blocks.json"
if not EMPTY_BLOCKS_FILE.exists():
    EMPTY_BLOCKS_FILE.write_bytes(b"[]")

In [3]:
UNSIGNED_REAL = r"(?:[0-9]+(?:[.][0-9]*)?|[.][0-9]+)"  # real number, unsigned
SIGNED_REAL = rf"[+-]?{UNSIGNED_REAL}"  # real number, signed

In [ ]:
ROTATION_PATTERN = re.compile(
    r"""
    \[\s*[αa]\s*\]                     # literal [α] or [a]
    (?:                                # capture rotation info
        [^D\(\[]*                      # run ahead to the next D
        D                              # sodium-D line (λ = 589 nm)
        [^\(\[]*                       # run ahead to the next parenthesis/bracket block
    )
    [\(\[]                             # opening parenthesis or bracket
        (?:                            # capture everything,
            (?:[^()\[\]]|\([^()]*\))*  # including nested parentheses
        )
    [\)\]]                             # closing parenthesis or bracket
    """,
    re.VERBOSE
)

In [5]:
E_VALUE = rf"({UNSIGNED_REAL})\s*%"
EE_PATTERN = (
    rf"(?:(?:>\s*)?{E_VALUE}\s*e\.?e|(?:e\.?e|enantiomeric excess)[:\.=>\s]*{E_VALUE})"
)
DE_PATTERN = rf"(?:(?:>\s*)?{E_VALUE}\s*d\.?e|(?:d\.?e|diastereomeric excess)[:\.=>\s]*{E_VALUE})"
EXCESS_PATTERN = re.compile(f"{EE_PATTERN}|{DE_PATTERN}", re.IGNORECASE)

In [6]:
R_VALUES = rf"({UNSIGNED_REAL})[\s\:\/]+({UNSIGNED_REAL})"
ER_PATTERN = (
    rf"(?:(?:>\s*)?{R_VALUES}\s*e\.?r|(?:e\.?r|enantiomeric ratio)[:\.=>\s]*{R_VALUES})"
)
DR_PATTERN = rf"(?:(?:>\s*)?{R_VALUES}\s*d\.?r|(?:d\.?r|diastereomeric ratio)[:\.=>\s]*{R_VALUES})"
RATIO_PATTERN = re.compile(rf"{ER_PATTERN}|{DR_PATTERN}", re.IGNORECASE)

In [7]:
FLOW_RATE = rf"""
(?i:
    {UNSIGNED_REAL}
    \s*mL
    \s*
    (?:
        / \s* min
        |
        min \s* -1
    )
)
"""

COLUMN_CLASS = r"(?i:chiral(?:cel|pak))"
COLUMN_TYPE = r"(?:A[DSYZ]|O[BDJXZ]|I[A-HJ-N])(?:-?[H3])?"
FULL_COLUMN = rf"(?:{COLUMN_CLASS}\s+{COLUMN_TYPE})"
COLUMN_PATTERN = rf"(?:{FULL_COLUMN}|{COLUMN_TYPE})"

HPLC_PATTERN = re.compile(
    rf"\b{COLUMN_PATTERN}[\s\-®]*{COLUMN_PATTERN}?\b[^\f]*?{FLOW_RATE}"
)
HPLC_PATTERN = re.compile(
    rf"""
    \b{COLUMN_PATTERN}[\s\-®]*{COLUMN_PATTERN}?\b
    .{{0,50}}?
    {FLOW_RATE}
    """,
    re.VERBOSE | re.DOTALL,
)

In [8]:
PATTERNS = {
    "rotation": ROTATION_PATTERN,
    "excess": EXCESS_PATTERN,
    "ratio": RATIO_PATTERN,
    "hplc": HPLC_PATTERN,
}

In [9]:
class Block(dict):
    def __init__(self, type, start, stop, text):
        super().__init__()
        self["type"] = type
        self["start"] = start
        self["stop"] = stop
        self["text"] = text


def find_data_blocks(text_file: pathlib.Path) -> list[Block]:
    full_text = text_file.read_text()
    blocks = []
    for name, pattern in PATTERNS.items():
        for match in pattern.finditer(full_text):
            blocks.append(Block(name, match.start(), match.end(), match.group(0)))
    blocks.sort(key=lambda b: b["start"])
    return blocks

In [10]:
with open(EMPTY_BLOCKS_FILE, "r") as f:
    empty_blocks = json.load(f)

text_files_to_process = []
detail_files = list(RETRIEVED_DIR.glob("**/details.json"))
for detail_file in tqdm(detail_files, desc="Analyzing details files"):
    with open(detail_file, "r") as f:
        details = json.load(f)
    article_id = str(details["id"])
    text_dir = EXTRACTED_DIR / article_id
    for file in details.get("files", []):
        stem = file["name"].split(".")[0]
        if stem in empty_blocks:
            continue
        text_file = text_dir / f"{stem}.txt"
        if text_file.exists() and text_file.stat().st_size > 0:
            blocks_file = BLOCKS_DIR / article_id / f"{stem}.json"
            if blocks_file.exists():
                continue
            text_files_to_process.append((stem, text_file, blocks_file))

print(f"Found {len(text_files_to_process)} text files to process")

saved_blocks = 0
for stem, text_file, blocks_file in tqdm(
    text_files_to_process, desc="Processing text files"
):
    blocks = find_data_blocks(text_file)
    if not blocks:
        empty_blocks.append(stem)
        continue
    blocks_file.parent.mkdir(parents=True, exist_ok=True)
    with open(blocks_file, "w") as f:
        json.dump(blocks, f, indent=2)
    saved_blocks += 1

with open(EMPTY_BLOCKS_FILE, "w") as f:
    json.dump(sorted(empty_blocks), f, indent=2)

print(f"Saved {saved_blocks} blocks")

Analyzing details files:   0%|          | 0/20585 [00:00<?, ?it/s]

Found 19996 text files to process


Processing text files:   0%|          | 0/19996 [00:00<?, ?it/s]

Saved 12660 blocks
